# EDA 11 — Loyers d'Annonce (Koumoul 2025)
**Source** : Koumoul / Data Fair | **Bronze** : `data/bronze/rentals/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "11_rentals"
draw_schema(
    "Bronze Schema — Loyers d'Annonce Koumoul 2025",
    [
        ("code_commune",   "str",   "Code INSEE commune (751xx)"),
        ("arrondissement", "int",   "Numero arrondissement (1-20)"),
        ("loyer_m2",       "float", "Loyer median predit (euros/m2/mois)"),
        ("loyer_m2_bas",   "float", "Borne basse de l'intervalle de confiance (euros/m2/mois)"),
        ("loyer_m2_haut",  "float", "Borne haute de l'intervalle de confiance (euros/m2/mois)"),
        ("nb_obs",         "int",   "Nombre d'annonces utilisees pour la prediction"),
        ("libgeo",         "str",   "Libelle geographique (Paris Xe Arrondissement)"),
        ("type_bien",      "str",   "Type de bien : Appartement"),
        ("ingested_at",    "datetime","Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
rental_files = sorted(glob.glob(f"{BRONZE_ROOT}/rentals/**/*.parquet", recursive=True))
if rental_files:
    df = pd.concat([pd.read_parquet(f) for f in rental_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    base = {1:35,2:33,3:32,4:36,5:34,6:40,7:42,8:43,9:32,10:29,11:30,12:27,13:26,14:29,15:31,16:39,17:34,18:26,19:22,20:23}
    df = pd.DataFrame([{"code_commune":f"751{a:02d}","arrondissement":a,
        "loyer_m2":base[a]+rng.normal(0,1.2),"loyer_m2_bas":base[a]-3+rng.normal(0,0.4),
        "loyer_m2_haut":base[a]+3+rng.normal(0,0.4),"nb_obs":int(rng.uniform(200,1500)),
        "libgeo":f"Paris {a}e Arrondissement","type_bien":"Appartement"} for a in range(1,21)])
print(f"Shape : {df.shape}")

In [ ]:
df_sorted = df.sort_values("loyer_m2",ascending=False).reset_index(drop=True)
norm = (df_sorted["loyer_m2"]-df_sorted["loyer_m2"].min())/(df_sorted["loyer_m2"].max()-df_sorted["loyer_m2"].min())
fig, ax = plt.subplots(figsize=(14,7))
ax.bar(range(len(df_sorted)),df_sorted["loyer_m2"],color=plt.cm.RdYlGn_r(norm),edgecolor="white",zorder=2)
ax.errorbar(range(len(df_sorted)),df_sorted["loyer_m2"],
    yerr=[df_sorted["loyer_m2"]-df_sorted["loyer_m2_bas"],df_sorted["loyer_m2_haut"]-df_sorted["loyer_m2"]],
    fmt="none",ecolor="#37474F",capsize=4,linewidth=1.2,zorder=3)
ax.set_xticks(range(len(df_sorted)))
ax.set_xticklabels(df_sorted["arrondissement"].astype(str),rotation=45)
ax.set_xlabel("Arrondissement (trie par loyer decroissant)")
ax.set_ylabel("Loyer median predit (euros/m2/mois)")
ax.set_title("Loyers d'annonce par arrondissement — Paris 2025 (IC 95%)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f} euros/m2"))
sm = plt.cm.ScalarMappable(cmap="RdYlGn_r",norm=plt.Normalize(df["loyer_m2"].min(),df["loyer_m2"].max()))
plt.colorbar(sm,ax=ax,label="euros/m2/mois")
save_fig("loyers_arrondissement", PREFIX); plt.show()
print(f"Loyer median Paris : {df['loyer_m2'].median():.1f} euros/m2/mois")
print(f"Ecart max-min : {df['loyer_m2'].max()-df['loyer_m2'].min():.1f} euros/m2")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df["loyer_m2"],bins=12,color=PALETTE["primary"],edgecolor="white",alpha=0.85)
axes[0].axvline(df["loyer_m2"].mean(),color=PALETTE["secondary"],linewidth=2,label=f"Moy : {df['loyer_m2'].mean():.1f} euros/m2")
axes[0].set_title("Distribution des loyers medians — Paris"); axes[0].set_xlabel("euros/m2/mois"); axes[0].legend()
ci_width = df["loyer_m2_haut"]-df["loyer_m2_bas"]
norm_ci = (ci_width-ci_width.min())/(ci_width.max()-ci_width.min())
axes[1].bar(df.sort_values("arrondissement")["arrondissement"].astype(str),ci_width.values,color=plt.cm.Blues(0.3+norm_ci.values*0.65),edgecolor="white")
axes[1].set_xlabel("Arrondissement"); axes[1].set_ylabel("Largeur IC 95% (euros/m2)"); axes[1].set_title("Incertitude de prediction par arrondissement")
plt.setp(axes[1].xaxis.get_majorticklabels(),rotation=45)
save_fig("distribution_et_incertitude", PREFIX); plt.show()